In [26]:
import osmnx as ox
import networkx as nx
import folium
from math import radians, sin, cos, sqrt, atan2

diem_dau = [10.761355552352372, 106.6683459252726]
diem_cuoi = [10.780211709469425, 106.69887295992045]

G = ox.graph_from_point(diem_dau, dist=6000, network_type='drive')

nut_dau = ox.distance.nearest_nodes(G, diem_dau[1], diem_dau[0])
nut_cuoi = ox.distance.nearest_nodes(G, diem_cuoi[1], diem_cuoi[0])

duong_dijkstra = nx.shortest_path(G, nut_dau, nut_cuoi, weight='length', method='dijkstra')

def khoang_cach_thang(lat1, lon1, lat2, lon2):
    R = 6371000
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

def heuristic(u, v):
    lat1 = G.nodes[u]['y']
    lon1 = G.nodes[u]['x']
    lat2 = G.nodes[v]['y']
    lon2 = G.nodes[v]['x']
    return khoang_cach_thang(lat1, lon1, lat2, lon2)

duong_astar = nx.astar_path(G, nut_dau, nut_cuoi, heuristic=heuristic, weight='length')

def lay_toa_do(duong):
    toa_do = []

    for nut in duong:
        vi_do = G.nodes[nut]['y']
        kinh_do = G.nodes[nut]['x']
        toa_do.append([vi_do, kinh_do])
    return toa_do

m = folium.Map(location=diem_dau, zoom_start=13)

folium.Marker(diem_dau, popup='UEH', tooltip = 'click').add_to(m)
folium.Marker(diem_cuoi, popup='Nha tho Duc Ba',  tooltip = 'click').add_to(m)

folium.PolyLine(lay_toa_do(duong_dijkstra), color='blue', weight=5, popup='Dijkstra').add_to(m)
folium.PolyLine(lay_toa_do(duong_astar), color='red', weight=3, popup='A*').add_to(m)

m

